# NB05 — Classification: Will This Stock Beat the Market?
**Machine Learning in Finance · Week 5**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB05_classification.ipynb)

**Finance question:** Instead of predicting the return, can we predict whether a stock will beat the S&P 500?

**Key idea:** **Classification** predicts a category (yes/no) and gives a **probability**. You choose a **threshold**: be picky (fewer calls, hopefully more right) or generous (catch more).

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Part A — Will it beat the index next month?
The label is **1 if the stock beat SPY** over the following period, **0 if it lagged**. Same 8 features as
NB04, trained on 2010–2017, tested on 2018–2025. Start by checking how often the answer is "yes".

In [ ]:
# 🟡 YOUR TURN
HORIZON = 1   # ← months ahead: 1 or 3. Is beating SPY over 3 months easier to predict?

panel = mlfin.make_features()
panel, LABEL = mlfin.add_beat_label(panel, months=HORIZON)
FEATURES = ["mom_12_1", "rev_1", "vol_12", "size", "btm", "ep", "beta_12", "max_1"]
train, test = mlfin.time_split(panel, "2017-12-31", gap_months=HORIZON)
X_train, y_train, _ = mlfin.prepare_xy(train, FEATURES, LABEL)
X_test, y_test, test_rows = mlfin.prepare_xy(test, FEATURES, LABEL)
print(f"Share of stock-months that beat SPY:  training {y_train.mean():.1%}   test {y_test.mean():.1%}")

*What to look for:* close to 50%, but **below** 50% in the test period: a few giant stocks drove the
index after 2018, so the typical stock lagged. A know-nothing model that always says "lag" would be right
about 52% of the time. That is the bar to beat.

## Step 1: logistic regression, "an S-curve that gives a probability"
Logistic regression combines the features into a score (like the line in NB03) and squeezes it through an
S-curve, so the output is always between 0 and 1: a **probability** of beating SPY.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logit = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
proba = logit.predict_proba(X_test)[:, 1]          # probability of "beat SPY" for every test stock-month
mlfin.plot_probabilities(proba, y_test, threshold=0.5)

*What to look for:* almost every probability sits between 45% and 55%. The model is **humble**: it never
claims to be sure. And the two colors (what actually happened) sit almost on top of each other — a first
warning that this is a hard problem.

## Step 2: the confusion matrix at threshold 0.5
Say "beat" when the probability is above the **threshold**. The 2×2 table counts the four outcomes.
In money terms: top-right = stocks you bought that lagged (money lost); bottom-right = winners you
bought (money made); bottom-left = winners you missed.

In [ ]:
# 🟡 YOUR TURN
THRESHOLD = 0.5   # ← try 0.52 or 0.53 (pickier) and 0.48 (more generous)

mlfin.plot_confusion(y_test, (proba > THRESHOLD).astype(int))
mlfin.classify_scores(y_test, proba, THRESHOLD).to_frame("logistic regression")

*How to read the scores:* **precision** = of the stocks we said would beat, the share that did.
**Recall** = of all the stocks that did beat, the share we caught. **AUC** ranks without any threshold:
0.5 is a coin flip, 1.0 is perfect.

## Step 3: the threshold trade-off
Move the threshold from generous (left) to picky (right). Recall always falls. **If the model has real
signal**, precision rises as you get pickier. Does it here?

In [ ]:
mlfin.plot_threshold_tradeoff(y_test, proba, threshold=THRESHOLD)

*What to look for:* the gold precision line hugs the dotted base-rate line (the wiggle at the far right
is a few dozen stocks: luck). Being pickier doesn't help when the model can't tell winners from losers. Keep this picture in mind for Part B,
where the model **does** have signal.

## Step 4: a decision tree — "the flowchart the machine writes"
A **decision tree** asks a yes/no question about one feature, then another, and so on. Each final box
(a leaf) gives a probability. `max_depth` is how many questions it may ask in a row.

In [ ]:
# 🟡 YOUR TURN
DEPTH = 3   # ← keep it at 2–4 so the drawing stays readable

from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(max_depth=DEPTH, random_state=0).fit(X_train, y_train)
mlfin.plot_tree(tree, FEATURES)

*How to read it:* start at the top box. If the condition is true, go left; otherwise right. The
`value` line shows the share of training stock-months that lagged and beat SPY in that box; the darker
the color, the more lopsided the box. Can you say each rule in plain English?

## Step 5: let the tree grow — memorizing the past

In [ ]:
# 🟡 YOUR TURN
BIG_DEPTH = 20   # ← try 5, 10, 20 and None (no limit)

big = DecisionTreeClassifier(max_depth=BIG_DEPTH, random_state=0).fit(X_train, y_train)
print(f"Depth {BIG_DEPTH}: {big.get_n_leaves():,} leaves (final boxes)")
pd.DataFrame({"training accuracy": [tree.score(X_train, y_train), big.score(X_train, y_train)],
              "test accuracy": [tree.score(X_test, y_test), big.score(X_test, y_test)]},
             index=[f"depth {DEPTH}", f"depth {BIG_DEPTH}"]).round(3)

*What to look for:* the deep tree scores far better on the past and **no better** on the future. With
thousands of leaves it has simply memorized which stock did what — the Week 1 game again.

## Part B — Spotting crashes: where classification shines
New question: **will this stock lose more than half its value in the next 12 months?** A risk manager
would love to know. Crashes are rare (about 1–2% of stock-months), so we tell the model to care about
the rare "crash" cases as much as the common ones (`class_weight="balanced"`).

In [ ]:
# 🟡 YOUR TURN
CRASH_FEATURES = ["vol_12", "mom_12_1", "size", "beta_12", "max_1", "leverage", "ep", "btm"]
# ← try removing "vol_12", or adding "margin" or "growth"

crash_panel = mlfin.add_crash_label(mlfin.make_features(), fall=0.5)
c_train, c_test = mlfin.time_split(crash_panel, "2017-12-31", gap_months=12)   # 12-month label → 12-month gap
Xc_train, yc_train, _ = mlfin.prepare_xy(c_train, CRASH_FEATURES, "crash_next_12")
Xc_test, yc_test, c_rows = mlfin.prepare_xy(c_test, CRASH_FEATURES, "crash_next_12")
crash_model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")).fit(Xc_train, yc_train)
crash_proba = crash_model.predict_proba(Xc_test)[:, 1]
print(f"Crash rate: training {yc_train.mean():.2%}   test {yc_test.mean():.2%}")
pd.Series(crash_model[-1].coef_[0], index=CRASH_FEATURES, name="weight (same scale)").round(2).sort_values().to_frame()

*What to look for:* the weights tell a finance story. Positive = more crash risk. Volatile, indebted
(high `leverage`), "cheap-looking" (high `btm`) stocks are flagged; why might each make sense?
Note the **12-month gap** between training and test: the label looks a year ahead.

In [ ]:
# 🟡 YOUR TURN
CRASH_THRESHOLD = 0.5   # ← try 0.7 and 0.9 (pickier)

mlfin.plot_confusion(yc_test, (crash_proba > CRASH_THRESHOLD).astype(int), labels=("No crash", "Crash"))
mlfin.plot_threshold_tradeoff(yc_test, crash_proba, threshold=CRASH_THRESHOLD,
                              title="Crash warnings: the threshold trade-off")
mlfin.classify_scores(yc_test, crash_proba, CRASH_THRESHOLD).to_frame("crash model")

*What to look for:* now the trade-off is real. A generous threshold catches most crashes but raises
many false alarms; a picky one flags fewer stocks and more of them actually crash. And notice:
a model that **never** warns would be 98% "accurate" — useless. Accuracy can lie when one outcome is rare.

## ⭐ GOING FURTHER (optional)
1. Ask an AI assistant: *"My classifier is 98% accurate at predicting stock crashes. Why might that be
   meaningless?"* Does its answer mention the base rate?
2. Change `fall=0.5` to `fall=0.3` (a 30% drop). Is a more common event easier or harder to predict?
3. Draw a depth-3 tree for the crash problem (`DecisionTreeClassifier(max_depth=3, class_weight="balanced")`
   with `mlfin.plot_tree(..., class_names=("No crash", "Crash"))`). Which question does it ask first?

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*